

## load required pacakages

In [3]:
import os,re, json, csv, joblib, time
import numpy as np
import pandas as pd
import sklearn
from joblib import Parallel, delayed
from pathlib import Path
from Bio.Seq import Seq
from Bio import SeqIO
from difflib import SequenceMatcher
from Bio.Data import CodonTable
from evcouplings.compare import DistanceMap
from scipy.optimize import check_grad
from scipy.sparse import issparse
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,precision_recall_curve, auc,roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, Lasso, RidgeCV,RidgeClassifierCV,Ridge,LogisticRegressionCV

In [ ]:
from data_loading import *
from data_preprocessing import *

In [858]:
## ------------------------
## GENE-LEVEL PIPELINE FOR TB RESISTANCE PREDICTION
## Author: Mahbuba Tasmin
## Description: Annotated end-to-end pipeline for extracting aligned sequences,
## translating to protein, handling frameshifts, and encoding features for phenotype modeling.
## ------------------------

# Step 1: Initialize gene(s) of interest

In [859]:
# Step 1: Initialize gene(s) of interest

# Define the candidate gene
# You can modify 'gene_name' to run the pipeline for a different gene
gene_name='inhA'
genes_of_interest=gene_name.split(',')
print("Gene in consideration", gene_name)

Gene in consideration inhA


In [860]:
# Set random seed for reproducibility
seed_everything(42)

# Step 2: Load phenotype and gene metadata

In [861]:
# Load phenotype data and other initializations
# phenotype_paths = ["../data/latest/master_table_resistance.csv","../data/catalog/cryptic_dataset.csv"]
phenotype_paths = ["../data/latest/master_resistance_table.csv"]

In [863]:
gene_details = pd.read_csv("../data/catalog/all_drug_genes_locations.csv")
# Filter the DataFrame for the genes of interest
filtered_df = gene_details [gene_details ['gene_name'].str.contains('|'.join(genes_of_interest), case=False, na=False)]
drug_name = 'ETHIONAMIDE' #filtered_df['drug_full'].values[0].upper()
print(f"The drug associated with the gene {gene_name} is: {drug_name}")


The drug associated with the gene inhA is: ETHIONAMIDE


In [864]:
filtered_df

,Unnamed: 0,Drug,gene_name,filename,role (hypothesized),references,Entry,Uniprot,Protein names,rv,drug_full,start_position_on_the_genomic_accession,end_position_on_the_genomic_accession,orientation
6,7,INH,inhA,inhA.fasta,NaN,"WHO 2021 catalog, category 1",P9WGR1,INHA_MYCTU,Enoyl-[acyl-carrier-protein] reductase [NADH] ...,Rv1484,isoniazid,1674201,1675010,plus


In [865]:
# Load data
print("creating data file for the drug", drug_name)
phenotype_data = load_phenotype_data(phenotype_paths,drug_name).reset_index()
print(phenotype_data.head(2))
# phenotype_data=phenotype_data.drop(['level_0'], axis=1)
phenotype_data["Isolate_mapped"] = [path.split("/")[-1].split(".vcf")[0].split(".cut")[0] for path in phenotype_data.path]
phenotype_mapping = dict(zip(phenotype_data['New_ID'], phenotype_data[drug_name.upper()]))

creating data file for the drug ETHIONAMIDE
Number of records in phenotype data: 2261
   index  Isolate AMIKACIN AMOXICILLIN AMOXICILLIN_CLAVULANATE CAPREOMYCIN  \
0      0  00R0025        R         NaN                     NaN           S   
1      1  00R0086        R         NaN                     NaN           R   

  CIPROFLOXACIN CLARITHROMYCIN CLOFAZIMINE CYCLOSERINE  ...  run run_combined  \
0           NaN            NaN         NaN         NaN  ...  NaN          NaN   
1           NaN              S         NaN         NaN  ...  NaN          NaN   

  bioproject biosample internal  \
0        NaN       NaN      NaN   
1        NaN       NaN      NaN   

                                                path Isolate_original  \
0  /n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...          00R0025   
1  /n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...          00R0086   

     accessions        New_ID Done  
0  SAMN05916422  SAMN05916422  1.0  
1  SAMN05918543  SAMN05918543  1.0 

/project/pi_annagreen_umass_edu/mahbuba/Data-Curation-for-MTB/protein-tasks/protein_translation/data_loading.py:11: DtypeWarning: Columns (2,3,6,7,8,11,16,19,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


# Step 3: Extract nucleotide sequences using reference alignment and gene coordinates

In [866]:
#define the fasta directory
fasta_dir = "../data/latest/fasta_files"
# fasta_dir = "../data/combined_fasta_files"

In [867]:
# second_row_df = filtered_df.iloc[[1]]
# print(second_row_df.head())
# drug_name = second_row_df['drug_full'].values[0].upper()
# print(drug_name)

In [890]:
# ------------------- Utilities ----------------------

def get_fasta_and_alignment(row, fasta_dir):
    # file_path = os.path.join(fasta_dir, row['filename']) ## add different row for rpob and rpoC
    file_path = os.path.join(fasta_dir, "inhA.fasta")
    alignment = load_alignment(file_path)
    return file_path, alignment

def get_h37rv_reference(alignment):
    return alignment.select(sequences=[alignment.id_to_index["MT_H37Rv"]])

def extract_sequence_from_alignment(h37rv_alignment):
    return ''.join(h37rv_alignment[0][h37rv_alignment[0] != "-"])

def extract_with_padding(h37rv_numbers, h37rv_ref, alignment, cds_start, cds_end, upstream_pad=100, downstream_pad=100):
    padded_start = cds_start - 0
    padded_end   = cds_end + 0

    subset_alignment, column_index, _, _ = sort_gene_indices(h37rv_numbers, padded_start, padded_end, alignment)
    h37rv_alignment = select_subset_alignment(h37rv_ref, padded_start, padded_end, h37rv_numbers[:, column_index])
    
    ungapped_seq = extract_sequence_from_alignment(h37rv_alignment)

    rel_start = upstream_pad
    rel_end   = rel_start + (cds_end - cds_start)
    return ungapped_seq[rel_start:rel_end], subset_alignment, h37rv_alignment


def extract_from_operon(h37rv_sequence_str, row, operon_start, special_shift=0):
    cds_start = int(row['start_position_on_the_genomic_accession'])
    cds_end   = int(row['end_position_on_the_genomic_accession'])

    rel_start = cds_start - operon_start + special_shift
    rel_end   = cds_end - operon_start + special_shift + 1
    return h37rv_sequence_str[rel_start:rel_end]

# ------------------- Main Loop ----------------------

subset_alignment = ''
h37rv_nongap_indices = []
h37rv_sequence_str = ''

# Load H37Rv genomic map once
h37rv_numbers = np.load("new_X_matrix_H37RV_coords_regenerated.npy")

for index, row in filtered_df.iterrows():
# for index, row in second_row_df.iterrows():
    gene_name = row['gene_name'].strip()
    print(f"\nProcessing: {gene_name} ({row['filename']})")

    # Load alignment
    file_path, alignment = get_fasta_and_alignment(row, fasta_dir)
    print(f"Gene shape: {alignment.matrix.shape}")

    # Extract reference
    h37rv_ref = get_h37rv_reference(alignment)

    # --- Special handling ---
    if gene_name in ['rpoB']:
        cds_start = int(row['start_position_on_the_genomic_accession'])
        cds_end   = int(row['end_position_on_the_genomic_accession'])
        # operon_start, operon_end = cds_start - 100, cds_end + 100
        operon_start, operon_end = 759706,  763325
        h37rv_sequence_str,subset_alignment, h37rv_alignment = extract_with_padding(h37rv_numbers, h37rv_ref, alignment, operon_start, operon_end)   
    # elif gene_name in ['embB']:
    #     # First extract full operon once (4239763–4249810)
    #     operon_start, operon_end = 4239763, 4249810
    #     subset_alignment, column_index, _, _ = sort_gene_indices(h37rv_numbers, operon_start, operon_end, alignment)
    #     h37rv_alignment = select_subset_alignment(h37rv_ref, operon_start, operon_end, h37rv_numbers[:, column_index])
    #     h37rv_sequence_str = extract_sequence_from_alignment(h37rv_alignment)

    #     # Then extract gene from full operon
    #     shift = 1 if gene_name == 'embB' else 0
    #     h37rv_sequence_str = extract_from_operon(h37rv_sequence_str, row, operon_start, special_shift=shift)

    # elif gene_name in ['gyrA', 'gyrB']:
    #     operon_start, operon_end = 5140, 9918
    #     subset_alignment, column_index, _, _ = sort_gene_indices(h37rv_numbers, operon_start, operon_end, alignment)
    #     h37rv_alignment = select_subset_alignment(h37rv_ref, operon_start, operon_end, h37rv_numbers[:, column_index])
    #     h37rv_sequence_str = extract_sequence_from_alignment(h37rv_alignment)
    #     h37rv_sequence_str = extract_from_operon(h37rv_sequence_str, row, operon_start)

    else:
        start_index = 1674201  #int(row['start_position_on_the_genomic_accession']) #1917939
        end_index   =  1675011 #int(row['end_position_on_the_genomic_accession']) # 1918745
        print(start_index in h37rv_numbers[0])  # Should be True
        print(end_index in h37rv_numbers[0])    # Should be True

        subset_alignment, column_index, _, _ = sort_gene_indices(h37rv_numbers, start_index, end_index, alignment)
        h37rv_alignment = select_subset_alignment(h37rv_ref, start_index, end_index, h37rv_numbers[:, column_index])
        h37rv_sequence_str  = extract_sequence_from_alignment(h37rv_alignment)
        print(h37rv_sequence_str)
    h37rv_nongap_indices = np.where(h37rv_alignment[0] != "-")[0]
    # -- Final reporting --
    print(f"Final extracted length: {len(h37rv_sequence_str)}")
    break



Processing: inhA (inhA.fasta)
Gene shape: (17943, 910)
False
False
Processed subset alignment for gene start 1674202 and end 1675011 in column 8
ATGACAGGACTGCTGGACGGCAAACGGATTCTGGTTAGCGGAATCATCACCGACTCGTCGATCGCGTTTCACATCGCACGGGTAGCCCAGGAGCAGGGCGCCCAGCTGGTGCTCACCGGGTTCGACCGGCTGCGGCTGATTCAGCGCATCACCGACCGGCTGCCGGCAAAGGCCCCGCTGCTCGAACTCGACGTGCAAAACGAGGAGCACCTGGCCAGCTTGGCCGGCCGGGTGACCGAGGCGATCGGGGCGGGCAACAAGCTCGACGGGGTGGTGCATTCGATTGGGTTCATGCCGCAGACCGGGATGGGCATCAACCCGTTCTTCGACGCGCCCTACGCGGATGTGTCCAAGGGCATCCACATCTCGGCGTATTCGTATGCTTCGATGGCCAAGGCGCTGCTGCCGATCATGAACCCCGGAGGTTCCATCGTCGGCATGGACTTCGACCCGAGCCGGGCGATGCCGGCCTACAACTGGATGACGGTCGCCAAGAGCGCGTTGGAGTCGGTCAACAGGTTCGTGGCGCGCGAGGCCGGCAAGTACGGTGTGCGTTCGAATCTCGTTGCCGCAGGCCCTATCCGGACGCTGGCGATGAGTGCGATCGTCGGCGGTGCGCTCGGCGAGGAGGCCGGCGCCCAGATCCAGCTGCTCGAGGAGGGCTGGGATCAGCGCGCTCCGATCGGCTGGAACATGAAGGATGCGACGCCGGTCGCCAAGACGGTGTGCGCGCTGCTGTCTGACTGGCTGCCGGCGACCACGGGTGACATCATCTACGCCGACGGCGGCGCGCACACCCAATTGCTCTAG
Final extracted length: 810


In [891]:
ref_seqs=pd.read_csv('../data/catalog/protein_sequences.csv')
ref_protein = ref_seqs.loc[ref_seqs['gene'] == gene_name, 'protein_sequence'].values[0]
ref_genome =  ref_seqs.loc[ref_seqs['gene'] == gene_name, 'genome_sequence'].values[0].upper()  

In [892]:
print(f"Extracted: {h37rv_sequence_str[:60]}...")
print(f"Expected : {ref_genome[:60]}...")
print(f"Match?    {h37rv_sequence_str == ref_genome}")


Extracted: ATGACAGGACTGCTGGACGGCAAACGGATTCTGGTTAGCGGAATCATCACCGACTCGTCG...
Expected : ATGACAGGACTGCTGGACGGCAAACGGATTCTGGTTAGCGGAATCATCACCGACTCGTCG...
Match?    True


In [893]:
pos = h37rv_sequence_str.find(ref_genome)
print(ref_genome in h37rv_sequence_str)

True


In [894]:
# from Bio import SeqIO

# seq = None
# for record in SeqIO.parse("/project/pi_annagreen_umass_edu/mahbuba/Data-Curation-for-MTB/protein-tasks/data/latest/fasta_files/tlyA.fasta", "fasta"):
#     if record.id == "MT_H37Rv":
#         seq = str(record.seq).replace("-", "")
#         break

# print(ref_genome in seq)
# pos = seq.find(ref_genome)
# print(pos)

In [895]:
ref_genome

'ATGACAGGACTGCTGGACGGCAAACGGATTCTGGTTAGCGGAATCATCACCGACTCGTCGATCGCGTTTCACATCGCACGGGTAGCCCAGGAGCAGGGCGCCCAGCTGGTGCTCACCGGGTTCGACCGGCTGCGGCTGATTCAGCGCATCACCGACCGGCTGCCGGCAAAGGCCCCGCTGCTCGAACTCGACGTGCAAAACGAGGAGCACCTGGCCAGCTTGGCCGGCCGGGTGACCGAGGCGATCGGGGCGGGCAACAAGCTCGACGGGGTGGTGCATTCGATTGGGTTCATGCCGCAGACCGGGATGGGCATCAACCCGTTCTTCGACGCGCCCTACGCGGATGTGTCCAAGGGCATCCACATCTCGGCGTATTCGTATGCTTCGATGGCCAAGGCGCTGCTGCCGATCATGAACCCCGGAGGTTCCATCGTCGGCATGGACTTCGACCCGAGCCGGGCGATGCCGGCCTACAACTGGATGACGGTCGCCAAGAGCGCGTTGGAGTCGGTCAACAGGTTCGTGGCGCGCGAGGCCGGCAAGTACGGTGTGCGTTCGAATCTCGTTGCCGCAGGCCCTATCCGGACGCTGGCGATGAGTGCGATCGTCGGCGGTGCGCTCGGCGAGGAGGCCGGCGCCCAGATCCAGCTGCTCGAGGAGGGCTGGGATCAGCGCGCTCCGATCGGCTGGAACATGAAGGATGCGACGCCGGTCGCCAAGACGGTGTGCGCGCTGCTGTCTGACTGGCTGCCGGCGACCACGGGTGACATCATCTACGCCGACGGCGGCGCGCACACCCAATTGCTCTAG'

In [896]:
len(ref_genome)

810

In [897]:
h37rv_sequence_str

'ATGACAGGACTGCTGGACGGCAAACGGATTCTGGTTAGCGGAATCATCACCGACTCGTCGATCGCGTTTCACATCGCACGGGTAGCCCAGGAGCAGGGCGCCCAGCTGGTGCTCACCGGGTTCGACCGGCTGCGGCTGATTCAGCGCATCACCGACCGGCTGCCGGCAAAGGCCCCGCTGCTCGAACTCGACGTGCAAAACGAGGAGCACCTGGCCAGCTTGGCCGGCCGGGTGACCGAGGCGATCGGGGCGGGCAACAAGCTCGACGGGGTGGTGCATTCGATTGGGTTCATGCCGCAGACCGGGATGGGCATCAACCCGTTCTTCGACGCGCCCTACGCGGATGTGTCCAAGGGCATCCACATCTCGGCGTATTCGTATGCTTCGATGGCCAAGGCGCTGCTGCCGATCATGAACCCCGGAGGTTCCATCGTCGGCATGGACTTCGACCCGAGCCGGGCGATGCCGGCCTACAACTGGATGACGGTCGCCAAGAGCGCGTTGGAGTCGGTCAACAGGTTCGTGGCGCGCGAGGCCGGCAAGTACGGTGTGCGTTCGAATCTCGTTGCCGCAGGCCCTATCCGGACGCTGGCGATGAGTGCGATCGTCGGCGGTGCGCTCGGCGAGGAGGCCGGCGCCCAGATCCAGCTGCTCGAGGAGGGCTGGGATCAGCGCGCTCCGATCGGCTGGAACATGAAGGATGCGACGCCGGTCGCCAAGACGGTGTGCGCGCTGCTGTCTGACTGGCTGCCGGCGACCACGGGTGACATCATCTACGCCGACGGCGGCGCGCACACCCAATTGCTCTAG'

In [898]:
# Compare with MycoBrowser CDS (optional)

match = SequenceMatcher(None, ref_genome ,h37rv_sequence_str).find_longest_match(0, len(h37rv_sequence_str), 0, len(ref_genome))
print(f"Longest match at ref_genome[{match.b}:{match.b + match.size}]")
print(f"Match length: {match.size} / {len(ref_genome)} ({match.size / len(ref_genome) * 100:.2f}%)")
print(f"Reference Sequence Length: {len(ref_genome)}")
print(f"Extracted Sequence Length: {len(h37rv_sequence_str)}")


Longest match at ref_genome[0:810]
Match length: 810 / 810 (100.00%)
Reference Sequence Length: 810
Extracted Sequence Length: 810


In [899]:
len(subset_alignment)

17943

In [900]:
#check alignment matrix shape
print(alignment.matrix.shape)
filenames = [path.split("/")[-1].split(".cut")[0] for path in subset_alignment.ids]
print("number of files", len(filenames))

filename_to_sequence = {}
for i, filename in enumerate(filenames):
    if filename not in filename_to_sequence and i < len(subset_alignment):
        filename_to_sequence[filename] = ''.join(subset_alignment[i])
filenames = list(filename_to_sequence.keys()) 

(17943, 910)
number of files 17943


# Step 4: Prepare sequence-phenotype CSV output

In [901]:
drug_name

'ETHIONAMIDE'

In [902]:
# Gene coordinate offsets relative to operon or padded extraction
operon_coords = {
    # 'gyrB':  (5240, 7267, 5140),
    # 'gyrA':  (7302, 9818, 5140),
    # 'embC': (4239863, 4243147, 4239763),
    # 'embA': (4243233, 4246517, 4239763),
    # 'embB': (4246513, 4249810, 4239763),
    'fabg1': (1673440, 1674183, 1673340),
    'rpoB': (759806, 763325, 759706)  # padded extraction for rpoB
}

output_file = f"../data/latest/sequence_data_csv/{gene_name}_{drug_name}_combined_sequence_data.csv"
data_list = []

for filename in filenames:
    if filename in phenotype_mapping and filename in filename_to_sequence:
        phenotype = phenotype_mapping[filename]
        sequence = filename_to_sequence[filename]

        if gene_name in operon_coords:
            gene_start, gene_end, operon_start = operon_coords[gene_name]
            start_offset = gene_start - operon_start
            end_offset   = gene_end   - operon_start + 1
            if gene_name == 'embB':
                start_offset += 1
                adjusted_indices = h37rv_nongap_indices[start_offset:end_offset]
                sequence = "".join(np.array(list(sequence))[adjusted_indices])
            else:
                # Apply slice to ungapped indices from H37Rv
                adjusted_indices = h37rv_nongap_indices[start_offset:end_offset]
                sequence = "".join(np.array(list(sequence))[adjusted_indices])
        else:
            # For non-operonic or simple genes
            sequence = "".join(np.array(list(sequence))[h37rv_nongap_indices])

        data_list.append([filename, sequence, phenotype, len(sequence)])

# Save the CSV
with open(output_file, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Filename", "Sequence", "Phenotype", "seq_len"])
    writer.writerows(data_list)

print(f"CSV file '{output_file}' saved successfully!")


CSV file '../data/latest/sequence_data_csv/inhA_ETHIONAMIDE_combined_sequence_data.csv' saved successfully!


# Step 5: Translate to protein and handle frame disruptions

In [903]:
file_path="../data/latest/sequence_data_csv/"+gene_name+"_"+ drug_name+ "_combined_sequence_data.csv"
gene_sequence_data=pd.read_csv(file_path)

In [904]:
print(gene_sequence_data.iloc[0])


Filename                                          SAMN05919664
Sequence     ATGACAGGACTGCTGGACGGCAAACGGATTCTGGTTAGCGGAATCA...
Phenotype                                                    S
seq_len                                                    810
Name: 0, dtype: object


In [905]:
seq1 = gene_sequence_data['Sequence'][0] 
seq2 = ref_genome


# Find longest matching subsequence (not just substring)
matcher = SequenceMatcher(None, seq2, seq1)
match = matcher.find_longest_match(0, len(seq2), 0, len(seq1))

print(f"Longest match at ref_genome[{match.a}:{match.a + match.size}]")
print(f"Match length: {match.size} / {len(seq1)} ({match.size / len(seq1):.2%})")
start_index = seq2.find(seq1)
print("If the match is from the start, index will be zero:", start_index)

Longest match at ref_genome[0:810]
Match length: 810 / 810 (100.00%)
If the match is from the start, index will be zero: 0


In [906]:
print("seq1[:60]:", seq1[:60])
print("ref_genome[:60]:", ref_genome[:60])
for i, (a, b) in enumerate(zip(seq1, seq2)):
    if a != b:
        print(f"Mismatch at position {i}: seq1={a}, seq2={b}")


seq1[:60]: ATGACAGGACTGCTGGACGGCAAACGGATTCTGGTTAGCGGAATCATCACCGACTCGTCG
ref_genome[:60]: ATGACAGGACTGCTGGACGGCAAACGGATTCTGGTTAGCGGAATCATCACCGACTCGTCG


In [907]:
non_matching = {}             # ← NEW dict

for _, row in gene_sequence_data.iterrows():      # cleaner than range(len())
    filename = row["Filename"]
    seq1     = row["Sequence"]

    # longest common *contiguous* subsequence (not global alignment)
    matcher = SequenceMatcher(None, ref_genome, seq1)
    match   = matcher.find_longest_match(0, len(ref_genome), 0, len(seq1))

    pct = match.size / len(seq1)                  # 1.00 → perfect, <1.00 → truncated/mutated

    if pct < 1.0:                                 # keep only imper- fect matches
        non_matching[filename] = pct              # store as 0.0–1.0 (or multiply by 100 for %)


In [908]:
len(non_matching)

228

In [909]:
len(gene_sequence_data)

2153

In [910]:
# --- sort the dict by pct, highest first ------------------------------
sorted_non_matching = dict(
    sorted(non_matching.items(), key=lambda kv: kv[1], reverse=True)
)

# --- count how many are ≥ 90 % ----------------------------------------
threshold = 0.9           # 90 %
high_pct = {k: v for k, v in non_matching.items() if v >= threshold}

print(f"Sequences ≥ 90 % match: {len(high_pct)} / {len(non_matching)}")


Sequences ≥ 90 % match: 0 / 228


In [911]:
def hamming_distance(seq1, seq2):
    return sum(c1 != c2 for c1, c2 in zip(seq1, seq2))

gene_sequence_data['hamming_dist'] = gene_sequence_data['Sequence'].apply(lambda x: hamming_distance(x, ref_genome))


In [912]:
# Custom translation function to deal with gaps, ambiguous bases, or internal stops
def translate_sequence_with_gaps(dna_seq, table="Standard", to_stop=False, handle_stops='remove', ref_protein_length=None):
    codon_table = CodonTable.unambiguous_dna_by_name[table]
    standard_table = codon_table.forward_table
    stop_codons = codon_table.stop_codons

    dna_seq = dna_seq.upper()
    protein_seq = []
    frameshift_mutations = False

    seq_len = len(dna_seq)
    i = 0
    cumulative_gap_count = 0

    while i + 3 <= seq_len:
        codon = dna_seq[i:i+3]
        if '-' in codon:
            # Codon contains gap(s)
            cumulative_gap_count += codon.count('-')
            protein_seq.append('-')
        elif re.search(r'[^ATCG]', codon):
            # Codon contains ambiguous nucleotide(s)
            protein_seq.append('X')
        else:
            if codon in stop_codons:
                if to_stop:
                    break
                else:
                    if handle_stops == 'remove':
                        pass  # Ignore stop codon
                    elif handle_stops == 'replace':
                        protein_seq.append('X')  # Replace stop codon with X
                    else:
                        protein_seq.append('*')  # Keep stop codon
            else:
                amino_acid = standard_table.get(codon)
                protein_seq.append(amino_acid)
        i += 3

    # Handle remaining nucleotides at the end
    if i < seq_len:
        remaining = dna_seq[i:]
        if '-' in remaining or re.search(r'[^ATCG]', remaining):
            protein_seq.append('-')
        else:
            # Remaining nucleotides less than a codon length (ignore instead of flagging)
            pass

    protein_str = ''.join(protein_seq)

    # **Fix: Prevent unnecessary frameshift flagging**
    if ref_protein_length is not None:
        translated_length = len(protein_seq)

        # Allow small mismatches (e.g., ±5% tolerance)
        length_difference = abs(translated_length - ref_protein_length)
        length_tolerance = max(1, int(ref_protein_length * 0.05))

        if length_difference > length_tolerance:
            frameshift_mutations = True  # Large length mismatch
        else:
            frameshift_mutations = False  # Small variations are fine

    # **Fix: Only flag internal stop codons, not the last one**
    if '*' in protein_str[:-1]:  # Ignore the last stop codon
        frameshift_mutations = True

    return protein_str, frameshift_mutations


In [913]:
def align_and_handle_deletions(translated_seq, ref_seq):
    """Handle deletions and ensure proper alignment with placeholders for gaps."""
    aligned_seq = []
    ref_index = 0
    trans_index = 0

    while ref_index < len(ref_seq) and trans_index < len(translated_seq):
        if translated_seq[trans_index] == ref_seq[ref_index]:
            # If they match, add the amino acid from the translated sequence
            aligned_seq.append(translated_seq[trans_index])
            trans_index += 1
        else:
            if translated_seq[trans_index] == '-':
                # If there is a gap in the translated sequence, mark it as a gap
                aligned_seq.append('-')
                trans_index += 1
            elif ref_seq[ref_index] == '-':
                # If the reference has a gap, skip the reference gap
                ref_index += 1
            else:
                # If it's a substitution (mismatch), append the amino acid from the translated sequence
                aligned_seq.append(translated_seq[trans_index])
                trans_index += 1

        # Move the reference index forward regardless
        ref_index += 1

    # **Fix: Trim excess gaps at the end**
    # aligned_seq_str = ''.join(aligned_seq).rstrip('-')
    aligned_seq_str = ''.join(aligned_seq)

    return aligned_seq_str


In [914]:
all_translations = []
all_labels = []
problematic_indices = set()
frameshift_mutations_list = []

for i in range(len(gene_sequence_data)):
    problematic = False
    h37rv_aa_str = Seq(h37rv_sequence_str).translate(to_stop=True)
    reference_length = len(h37rv_aa_str)
    # Correct for CDS
    translation, frameshift = translate_sequence_with_gaps(gene_sequence_data["Sequence"][i], to_stop=False, ref_protein_length=reference_length)
    # dna fasta files already took care of orientation so we skip that step here.
    aligned_translation = align_and_handle_deletions(translation, h37rv_aa_str)


    if i == 1:
        print("translation before aligning", translation)
        print("translation after aligning", aligned_translation)
        print(f"Translated length: {len(translation)}, Reference length: {reference_length}")
        print(f"Final aligned sequence: {aligned_translation}")

    if frameshift:
        problematic = True
        problematic_indices.add(i)
        frameshift_mutations_list.append(1)
        aligned_translation = '0' * reference_length
    else:
        frameshift_mutations_list.append(0)

    # print(f"Row {row_index}: frameshift={frameshift}, aligned_translation={aligned_translation[:50]}...")
    all_translations.append(aligned_translation)
    all_labels.append(gene_sequence_data["Phenotype"][i])

# **Add new columns to the DataFrame**
gene_sequence_data["Protein_Sequence"] = all_translations
gene_sequence_data["Frameshift_Mutation"] = frameshift_mutations_list

translation before aligning MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRLPAKAPLLELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTLAMSAIVGGALGEEAGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL
translation after aligning MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRLPAKAPLLELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTLAMSAIVGGALGEEAGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL
Translated length: 269, Reference length: 269
Final aligned sequence: MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRLPAKAPLLELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTLAMSAIVGGALGEEAGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL


### add protein sequence to the gene csv file and save

In [915]:
output_file_name=gene_name+"_"+ drug_name+ "_combined_sequence_data.csv"
gene_sequence_data.to_csv("../data/latest/sequence_data_csv/"+output_file_name, index=False)

In [916]:
non_matching = {}             # ← NEW dict

for _, row in gene_sequence_data.iterrows():      # cleaner than range(len())
    filename = row["Filename"]
    seq1     = row["Protein_Sequence"]

    # longest common *contiguous* subsequence (not global alignment)
    matcher = SequenceMatcher(None, ref_protein, seq1)
    match   = matcher.find_longest_match(0, len(ref_protein), 0, len(seq1))

    pct = match.size / len(seq1)                  # 1.00 → perfect, <1.00 → truncated/mutated

    if pct < 1.0:                                 # keep only imper- fect matches
        non_matching[filename] = pct              # store as 0.0–1.0 (or multiply by 100 for %)


In [917]:
# --- sort the dict by pct, highest first ------------------------------
sorted_non_matching = dict(
    sorted(non_matching.items(), key=lambda kv: kv[1], reverse=True)
)

# --- count how many are ≥ 90 % ----------------------------------------
threshold = 0.90           # 90 %
high_pct = {k: v for k, v in non_matching.items() if v >= threshold}

print(f"Sequences ≥ 90 % match: {len(high_pct)} / {len(non_matching)}")

Sequences ≥ 90 % match: 41 / 209


In [918]:
ref_protein

'MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRLPAKAPLLELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTLAMSAIVGGALGEEAGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL'

In [919]:
# Find longest matching subsequence (not just substring)
seq1 = gene_sequence_data['Protein_Sequence'][0] 
seq2 = ref_protein


matcher = SequenceMatcher(None, seq2, seq1)
match = matcher.find_longest_match(0, len(seq2), 0, len(seq1))

print(f"Longest match at ref_protein[{match.a}:{match.a + match.size}]")
print(f"Longest Match length: {match.size} / {len(seq1)} ({match.size / len(seq1):.2%})")

# Show matching region
print("Matched ref segment:", ref_protein[match.a:match.a + match.size])
print("Matched seq1 segment:", seq1[match.b:match.b + match.size])


print(f"Match ratio: {matcher.ratio() * 100:.2f}%")

Longest match at ref_protein[0:269]
Longest Match length: 269 / 269 (100.00%)
Matched ref segment: MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRLPAKAPLLELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTLAMSAIVGGALGEEAGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL
Matched seq1 segment: MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRLPAKAPLLELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTLAMSAIVGGALGEEAGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL
Match ratio: 100.00%


In [920]:
from Bio import pairwise2

alignments = pairwise2.align.globalxx(ref_protein, seq1)
top = alignments[0]
print(pairwise2.format_alignment(*top))


MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRLPAKAPLLELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTLAMSAIVGGALGEEAGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL
|||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRLPAKAPLLELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTLAMSAIVGGALGEEAGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL
  Score=269



In [921]:
## explore and confirm translated outputs for feature and matrix.
unique_elements, counts = np.unique(frameshift_mutations_list, return_counts=True)
value_counts_dict = dict(zip(unique_elements, counts))
print(value_counts_dict)

problem_percentage = (len(problematic_indices) / len(gene_sequence_data)) * 100
print(f"Problematic percentage: {problem_percentage}%")

print("Keeping all sequences.")
valid_indices = range(len(gene_sequence_data))

valid_translations = [(filenames[i], all_translations[i][:reference_length]) for i in valid_indices]
valid_labels = [all_labels[i] for i in valid_indices]

valid_lengths = [len(seq[1]) for seq in valid_translations]
length_mismatches = [i for i, length in enumerate(valid_lengths) if length != reference_length]
total_sequences = len(valid_translations)
num_mismatches = len(length_mismatches)
print(f"Reference length expected: {reference_length}")
print(f"Sample of sequence lengths after truncation: {[len(seq[1]) for seq in valid_translations[:10]]}")

if num_mismatches == 0:
    print("All translations have the correct length.")
else:
    mismatch_percentage = (num_mismatches / total_sequences) * 100
    print(f"{num_mismatches} sequences have varying lengths out of {total_sequences} sequences.")
    print(f"Percentage of sequences with varying lengths: {mismatch_percentage:.2f}%")

print(f"Total sequences after filtering: {len(valid_translations)}")
print(f"Sample of sequence lengths after filtering: {[len(seq[1]) for seq in valid_translations[:10]]}")

{0: 2153}
Problematic percentage: 0.0%
Keeping all sequences.
Reference length expected: 269
Sample of sequence lengths after truncation: [269, 269, 269, 269, 269, 269, 269, 269, 269, 269]
All translations have the correct length.
Total sequences after filtering: 2153
Sample of sequence lengths after filtering: [269, 269, 269, 269, 269, 269, 269, 269, 269, 269]


In [922]:
#check with ref protein again
start_index = ref_protein.find(gene_sequence_data['Protein_Sequence'][0])
start_index

0

In [923]:
gene_sequence_data.head()

,Filename,Sequence,Phenotype,seq_len,hamming_dist,Protein_Sequence,Frameshift_Mutation
0,SAMN05919664,ATGACAGGACTGCTGGACGGCAAACGGATTCTGGTTAGCGGAATCA...,S,810,0,MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRL...,0
1,SAMN05919631,ATGACAGGACTGCTGGACGGCAAACGGATTCTGGTTAGCGGAATCA...,S,810,0,MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRL...,0
2,SAMEA3404180,ATGACAGGACTGCTGGACGGCAAACGGATTCTGGTTAGCGGAATCA...,R,810,0,MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRL...,0
3,SAMN02419685,ATGACAGGACTGCTGGACGGCAAACGGATTCTGGTTAGCGGAATCA...,R,810,0,MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRL...,0
4,SAMN06010193,ATGACAGGACTGCTGGACGGCAAACGGATTCTGGTTAGCGGAATCA...,R,810,0,MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRL...,0


# Step 6: write aa fasta file with filename, aa seq, phenotype and frameshift flag

In [924]:

def write_fasta_with_metadata_from_df(df, output_file, reference_length):
    """
    Writes translated protein sequences to a FASTA file with metadata in the header.

    Args:
        df (pd.DataFrame): DataFrame containing Filename, Protein_Sequence, Phenotype, and Frameshift_Mutation.
        output_file (str): Path to save the FASTA file.
        reference_length (int): Length of the reference protein sequence.
    """
    with open(output_file, "w") as fasta_file:
        for _, row in df.iterrows():
            filename = row["Filename"]
            sequence = row["Protein_Sequence"] 
            phenotype = row["Phenotype"]
            frameshift_flag = row["Frameshift_Mutation"]
            seq_len = row["seq_len"]
            

            # Construct FASTA header
            header = f">{filename} | {phenotype} | {seq_len} | Frameshift: {frameshift_flag}"
            fasta_file.write(header + "\n")
            fasta_file.write(sequence + "\n")


In [925]:
# Define output file path
protein_name = f'../data/latest/aa_fasta/all_sequences_{gene_name}_{drug_name}_aa.fasta'

# Ensure the DataFrame has the correct columns
if {"Filename", "Protein_Sequence", "Phenotype", "Frameshift_Mutation"}.issubset(gene_sequence_data.columns):
    write_fasta_with_metadata_from_df(gene_sequence_data, protein_name, reference_length)
    print(f"FASTA file saved at {protein_name}")
else:
    print("Error: The DataFrame is missing required columns.")


FASTA file saved at ../data/latest/aa_fasta/all_sequences_inhA_ETHIONAMIDE_aa.fasta


In [926]:

filtered_translations = [(fn, seq) for (fn, seq) in valid_translations if len(seq) == reference_length]

filtered_labels = [valid_labels[i] for i, (_, seq) in enumerate(valid_translations) if len(seq) == reference_length]

filtered_flags = [frameshift_mutations_list[i] for i, (_, seq) in enumerate(valid_translations) if len(seq) == reference_length]


print(f"Reference length expected: {reference_length}")
print(f"Total sequences before filtering: {len(all_translations)}")
print(f"Total sequences after filtering: {len(filtered_translations)}")
print(f"Dropped {len(all_translations) - len(filtered_translations)} sequences.")



Reference length expected: 269
Total sequences before filtering: 2153
Total sequences after filtering: 2153
Dropped 0 sequences.


# Step 7: One-hot encode mutated positions relative to reference protein
## Mutation = 1, match = 0

In [927]:
def convert_to_onehot_with_reference(aa_seq, ref_aa):
    # return np.array([1 if aa == ref else 0 for aa, ref in zip(aa_seq, ref_aa)])
    return np.array([0 if aa == ref else 1 for aa, ref in zip(aa_seq, ref_aa)])

def encode_sequence(sequence, reference_length, h37rv_aa_str):
    encoded = convert_to_onehot_with_reference(str(sequence), str(h37rv_aa_str))
    return encoded

In [928]:

# --------- Step 2: Encode Sequences ---------

num_cores = joblib.cpu_count()

encoded_features = Parallel(n_jobs=num_cores)(
    delayed(encode_sequence)(seq, reference_length, h37rv_aa_str)
    for _, seq in filtered_translations
)

# Sanity check
lengths = [len(f) for f in encoded_features]
assert all(l == reference_length for l in lengths), "Inconsistent feature lengths after encoding!"
print(f"Encoding complete. Unique encoded lengths: {set(lengths)}")


Encoding complete. Unique encoded lengths: {269}


# Step 8: Save feature matrix for ML training
## Shape: (n_samples, 1 + L), where L = reference protein length
## First column = binary mutation flag (e.g. frame-shift)

In [929]:

# --------- Step 3: Assemble Feature Matrix ---------

mutation_flags = np.array(filtered_flags).reshape(-1, 1)
encoded_matrix = np.array(encoded_features)
feature_matrix = np.column_stack((encoded_matrix,mutation_flags))

print(f"Final feature matrix shape: {feature_matrix.shape}")

# --------- Step 4: Save ---------

np.save(f'../data/latest/feature_matrix_labels/{gene_name}_{drug_name}_feature_matrix.npy', feature_matrix)
np.save(f'../data/latest/feature_matrix_labels/{gene_name}_{drug_name}_labels.npy', filtered_labels)
print("Full preprocessing pipeline complete.")

Final feature matrix shape: (2153, 270)
Full preprocessing pipeline complete.


# Step 9: Sample Data loading for Model Training

In [930]:
def load_feature_matrix_and_labels(gene_name):
    file_dir ="../data/latest/feature_matrix_labels"
    feature_matrix_file = f'{file_dir}/{gene_name}_{drug_name}_feature_matrix.npy'
    print(feature_matrix_file)
    print(os.path.exists(feature_matrix_file))
    labels_file = f'{file_dir}/{gene_name}_{drug_name}_labels.npy'
    
    if os.path.exists(feature_matrix_file) and os.path.exists(labels_file):
        print(f"Loading feature matrix and labels for {gene_name} from disk.")
        feature_matrix = np.load(feature_matrix_file)
        labels = np.load(labels_file)
    else:
        print(f"Need to create feature matrix and labels for {gene_name}.")
        # feature_matrix, labels = create_feature_matrix(subset_alignment, drug, phenotype_data, h37rv_nongap_indices, h37rv_sequence_str, orientation, gene_name, discard)
        
    return feature_matrix, labels      

In [931]:
X, y = load_feature_matrix_and_labels(gene_name)

../data/latest/feature_matrix_labels/inhA_ETHIONAMIDE_feature_matrix.npy
True
Loading feature matrix and labels for inhA from disk.


In [932]:
# try with full sequence first
y_encoded = encode_labels(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)
print(f"Train data shape before deduplication: {np.array(X_train).shape}")
full_train_shape = np.array(X_train).shape


lasso = LassoCV(max_iter=10000, cv=5, random_state=42, alphas=[0.001, 0.01, 0.1, 1, 10, 100])
# lasso = LassoCV()
lasso.fit(X_train,y_train)

regular_lasso_R2 = lasso.score(X_test, y_test)
print("Lasso Score", regular_lasso_R2)
# logging.info(f"Lasso Score: {regular_lasso_R2}")
regular_lasso_auc =sklearn.metrics.roc_auc_score(y_test, lasso.predict(X_test))
print("Lasso AUC", regular_lasso_auc)
# logging.info(f"Lasso AUC: {regular_lasso_auc}")
regular_lasso_mse = mean_squared_error(y_test, lasso.predict(X_test))
print(f"regular lasso mse:{regular_lasso_mse}")


ridge = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10]).fit(X_train, y_train)
full_ridge_score = ridge.score(X_test, y_test)
full_ridge_auc = sklearn.metrics.roc_auc_score(y_test, ridge.predict(X_test))
full_ridge_mse = mean_squared_error(y_test, ridge.predict(X_test))
print("ridge score", full_ridge_score)
print("ridge mse", full_ridge_mse)
print("ridge auc", full_ridge_auc)

logistic_model = LogisticRegressionCV(cv=3, scoring="roc_auc", max_iter=5000,Cs=[1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100], class_weight="balanced").fit(X_train, y_train)
log_score = logistic_model.score(X_test, y_test)
log_auc = sklearn.metrics.roc_auc_score(y_test, logistic_model.predict(X_test))
log_mse = mean_squared_error(y_test, logistic_model.predict(X_test))

print("log score", log_score)
print("log mse", log_mse)
print("log auc", log_auc)


Train data shape before deduplication: (1722, 270)
Lasso Score 0.11007902479376963
Lasso AUC 0.6055423653361033
regular lasso mse:0.19370623926167888
ridge score 0.10861102186436233
ridge mse 0.1940257747424722
ridge auc 0.6055423653361033
log score 0.6056165603205224
log mse 0.26450116009280744
log auc 0.6042068556165603


## generate combined matrix

In [933]:
import numpy as np
import pandas as pd
import os
from Bio.Seq import Seq
from difflib import SequenceMatcher
from joblib import Parallel, delayed

# -----------------------------
# CONFIGURATION
# -----------------------------
DATA_DIR = "../data/latest/sequence_data_csv"
OUT_DIR = "../data/latest/feature_matrix_labels"
REFERENCE_SEQ_FILE = "../data/catalog/protein_sequences.csv"

# -----------------------------
# UTILITIES
# -----------------------------
def hamming_distance(seq1, seq2):
    return sum(c1 != c2 for c1, c2 in zip(seq1, seq2))

def convert_to_onehot_with_reference(aa_seq, ref_aa):
    return np.array([0 if aa == ref else 1 for aa, ref in zip(aa_seq, ref_aa)])

def encode_sequence(sequence, reference):
    return convert_to_onehot_with_reference(str(sequence), str(reference))

def get_common_identifiers(gene_dfs):
    shared = set(gene_dfs[0]['Filename'])
    for df in gene_dfs[1:]:
        shared &= set(df['Filename'])
    return list(shared)

# -----------------------------
# MAIN PIPELINE
# -----------------------------
def generate_combined_feature_matrix(gene_names, output_prefix):
    ref_seqs = pd.read_csv(REFERENCE_SEQ_FILE)
    gene_dfs = []
    ref_proteins = {}

    # Load data per gene
    for gene in gene_names:
        file_path = os.path.join(DATA_DIR, f"{gene}_{drug_name}_combined_sequence_data.csv")
        df = pd.read_csv(file_path)
        df = df[df['Frameshift_Mutation'] == 0].copy()
        gene_dfs.append(df)

        ref_protein = ref_seqs.loc[ref_seqs['gene'] == gene, 'protein_sequence'].values[0]
        ref_proteins[gene] = ref_protein

    # Identify shared isolates
    common_ids = get_common_identifiers(gene_dfs)
    print(f"Number of shared isolates: {len(common_ids)}")

    all_vectors = []
    all_labels = []

    for idx in common_ids:
        row_vectors = []
        skip = False
        for i, gene in enumerate(gene_names):
            df = gene_dfs[i]
            row = df[df['Filename'] == idx].iloc[0]
            protein_seq = row['Protein_Sequence']
            ref_protein = ref_proteins[gene]
            try:
                encoded = np.asarray(encode_sequence(protein_seq, ref_protein)).flatten()
                if encoded.ndim != 1:
                    raise ValueError("Encoded vector is not 1D.")
                row_vectors.append(encoded)
            except Exception as e:
                print(f"Skipping {idx} due to vector problem in gene {gene}: {e}")
                skip = True
                break
        if skip:
            continue
        try:
            full_vector = np.concatenate(row_vectors)
            all_vectors.append(full_vector)
            all_labels.append(row['Phenotype'])
        except Exception as e:
            print(f"Skipping {idx} due to concatenation failure: {e}")

    expected_len = len(all_vectors[0])
    filtered = [(v, l) for v, l in zip(all_vectors, all_labels) if len(v) == expected_len]
    all_vectors, all_labels = zip(*filtered)
    X = np.stack(all_vectors)
    y = pd.Series(all_labels).astype("category").cat.codes.values


    # Save to disk
    os.makedirs(OUT_DIR, exist_ok=True)
    np.save(os.path.join(OUT_DIR, f"{output_prefix}_feature_matrix.npy"), X)
    np.save(os.path.join(OUT_DIR, f"{output_prefix}_labels.npy"), y)

    print(f"Saved combined matrix: {X.shape}, labels: {y.shape}")
    return X, y


In [935]:

genes = ["ethA","ethR","inhA"]  a# or more
drug_name="ETHIONAMIDE"
generate_combined_feature_matrix(genes, output_prefix="ethA_ethR_inhA_ETH")

Number of shared isolates: 2153
Saved combined matrix: (94, 973), labels: (94,)


(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]),
 array([0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1,
        1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0,
        0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1,
        0, 1, 0, 1, 1, 1], dtype=int8))

In [15]:
import numpy as np
import pandas as pd
import os
from Bio.Seq import Seq
from difflib import SequenceMatcher
from joblib import Parallel, delayed

# -----------------------------
# CONFIGURATION
# -----------------------------
DATA_DIR = "../data/latest/sequence_data_csv"
OUT_DIR = "../data/latest/feature_matrix_labels"
REFERENCE_SEQ_FILE = "../data/catalog/protein_sequences.csv"

# -----------------------------
# UTILITIES
# -----------------------------
def hamming_distance(seq1, seq2):
    return sum(c1 != c2 for c1, c2 in zip(seq1, seq2))

def convert_to_onehot_with_reference(aa_seq, ref_aa):
    return np.array([0 if aa == ref else 1 for aa, ref in zip(aa_seq, ref_aa)])

# def encode_sequence(sequence, reference):
#     return convert_to_onehot_with_reference(str(sequence), str(reference))

def encode_sequence(sequence, reference):
    sequence = str(sequence)
    reference = str(reference)
    if len(sequence) == len(reference):
        return convert_to_onehot_with_reference(sequence, reference)
    elif len(sequence) == len(reference) - 1:
        # Assume last residue matches reference
        padded_sequence = sequence + reference[-1]
        return convert_to_onehot_with_reference(padded_sequence, reference)
    else:
        raise ValueError(f"Sequence length {len(sequence)} doesn't match reference length {len(reference)} or reference-1")


def get_common_identifiers(gene_dfs):
    shared = set(gene_dfs[0]['Filename'])
    for df in gene_dfs[1:]:
        shared &= set(df['Filename'])
    return list(shared)

def generate_combined_feature_matrix(gene_names, output_prefix):
    ref_seqs = pd.read_csv(REFERENCE_SEQ_FILE)
    gene_dfs = []
    ref_proteins = {}

    # Load data per gene
    for gene in gene_names:
        file_path = os.path.join(DATA_DIR, f"{gene}_{drug_name}_combined_sequence_data.csv")
        df = pd.read_csv(file_path)
        df = df[df['Frameshift_Mutation'] == 0].copy()
        gene_dfs.append(df)

        ref_protein = ref_seqs.loc[ref_seqs['gene'] == gene, 'protein_sequence'].values[0]
        ref_proteins[gene] = ref_protein

    # Identify shared isolates
    common_ids = get_common_identifiers(gene_dfs)
    print(f"Number of shared isolates: {len(common_ids)}")

    expected_lengths = {gene: len(ref_proteins[gene]) for gene in gene_names}
    print("Expected lengths per gene:", expected_lengths)

    final_vectors = []
    final_labels = []

    skip_log = []

    for idx in common_ids:
        row_vectors = []
        skip = False
        skip_reason = ""

        for i, gene in enumerate(gene_names):
            df = gene_dfs[i]
            try:
                row = df[df['Filename'] == idx].iloc[0]
                protein_seq = row['Protein_Sequence']
                ref_protein = ref_proteins[gene]

                encoded = encode_sequence(protein_seq, ref_protein)
                if len(encoded) != expected_lengths[gene]:
                    raise ValueError(f"Unexpected length {len(encoded)} for gene {gene} (expected {expected_lengths[gene]})")

                row_vectors.append(encoded)

            except Exception as e:
                skip = True
                skip_reason = f"{gene}: {str(e)}"
                break

        if skip:
            skip_log.append({"Filename": idx, "Reason": skip_reason})
            continue

        try:
            full_vector = np.concatenate(row_vectors)
            final_vectors.append(full_vector)
            final_labels.append(row['Phenotype'])
        except Exception as e:
            skip_log.append({"Filename": idx, "Reason": f"Concatenation error: {e}"})

    if len(final_vectors) == 0:
        raise RuntimeError("No valid isolates found after filtering.")

    X = np.stack(final_vectors)
    y = pd.Series(final_labels).astype("category").cat.codes.values

    # Save matrix
    os.makedirs(OUT_DIR, exist_ok=True)
    np.save(os.path.join(OUT_DIR, f"{output_prefix}_feature_matrix.npy"), X)
    np.save(os.path.join(OUT_DIR, f"{output_prefix}_labels.npy"), y)

    # Save report
    report_df = pd.DataFrame(skip_log)
    report_path = os.path.join(OUT_DIR, f"{output_prefix}_skipped_report.csv")
    report_df.to_csv(report_path, index=False)

    print(f"Saved combined matrix: {X.shape}, labels: {y.shape}")
    print(f"Skipped {len(skip_log)} isolates. Report saved to: {report_path}")

    return X, y


In [22]:
genes = ["rpsL","gid"]  # or more
drug_name="STREPTOMYCIN"
generate_combined_feature_matrix(genes, output_prefix="STREPTOMYCIN")

Number of shared isolates: 7682
Expected lengths per gene: {'rpsL': 124, 'gid': 224}
Saved combined matrix: (7682, 348), labels: (7682,)
Skipped 0 isolates. Report saved to: ../data/latest/feature_matrix_labels/STREPTOMYCIN_skipped_report.csv


(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]),
 array([0, 0, 1, ..., 1, 0, 1], dtype=int8))